# 🍎 Pendeteksi Kematangan Buah — Notebook Eksplorasi

Notebook ini digunakan untuk:
1. Eksplorasi dataset
2. Visualisasi distribusi kelas
3. Melatih model
4. Evaluasi dan interpretasi hasil
5. Grad-CAM visualisasi

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Persiapan Dataset

In [ ]:
# Jalankan setelah dataset Kaggle didownload dan diekstrak
# Ganti PATH_DATASET dengan lokasi folder dataset Anda

PATH_DATASET = '/path/ke/dataset/fruits-quality-fresh-vs-rotten'

!python src/prepare_data.py --src {PATH_DATASET} --ratio 0.7 0.15 0.15

## 2. Eksplorasi Dataset

In [ ]:
from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR
from src.dataset import get_datasets
from src.utils import class_distribution, plot_class_distribution, visualize_dataset_samples

train_ds, val_ds, test_ds, class_names = get_datasets()
print(f'\nJumlah kelas: {len(class_names)}')
print('Nama kelas:', class_names)

In [ ]:
# Distribusi kelas di training set
counts = class_distribution(TRAIN_DIR, class_names)
plot_class_distribution(counts, title='Distribusi Kelas — Train Set')

In [ ]:
# Contoh gambar dari training set
visualize_dataset_samples(train_ds, class_names, n=16)

## 3. Training Model

In [ ]:
# Jalankan training (bisa memakan waktu 30-60 menit tergantung GPU)
# Alternatif: jalankan dari terminal: python src/train.py

from src.model import build_model, unfreeze_for_finetuning, model_summary

num_classes = len(class_names)
model = build_model(num_classes)
model_summary(model)

In [ ]:
# Untuk training penuh, gunakan script train.py
# Di sini contoh training singkat 5 epoch
from src.config import OUTPUT_DIR

history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds,
    verbose=1
)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()

## 4. Evaluasi Model

In [ ]:
# Evaluasi menggunakan model terbaik yang sudah dilatih
!python src/evaluate.py

## 5. Prediksi Satu Gambar + Grad-CAM

In [ ]:
import json
from PIL import Image
from src.config import MODEL_DIR, IMG_SIZE
from src.utils import grad_cam, overlay_grad_cam

# Ganti dengan path gambar buah Anda
IMG_PATH = 'data/test/fresh_apple/R0_0.jpg'

# Muat model
best_model = tf.keras.models.load_model(f'{MODEL_DIR}/best_model.keras')
with open(f'{MODEL_DIR}/class_names.json') as f:
    cnames = json.load(f)

# Preprocess
img = Image.open(IMG_PATH).convert('RGB').resize(IMG_SIZE)
arr = np.array(img, dtype=np.float32)
arr_pp = tf.keras.applications.mobilenet_v2.preprocess_input(arr.copy())
arr_pp = np.expand_dims(arr_pp, 0)

# Prediksi
proba = best_model.predict(arr_pp, verbose=0)[0]
best  = np.argmax(proba)
print(f'Prediksi: {cnames[best]} ({proba[best]*100:.2f}%)')
print('Segar' if 'fresh' in cnames[best].lower() else 'Busuk')

In [ ]:
# Grad-CAM visualisasi
heatmap = grad_cam(best_model, arr_pp, layer_name='out_relu')
overlay = overlay_grad_cam(IMG_PATH, heatmap)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(arr.astype('uint8')); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Heatmap'); axes[1].axis('off')
axes[2].imshow(overlay);             axes[2].set_title('Grad-CAM'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 6. Jalankan Aplikasi Web

In [ ]:
# Jalankan antarmuka Gradio (dari terminal lebih disarankan)
# !python app/app.py --port 7860

# Atau inline di notebook:
import subprocess
# subprocess.Popen(['python', 'app/app.py'])
print('Untuk menjalankan app: python app/app.py')
print('Buka browser: http://localhost:7860')